In [1]:
%pip install langgraph
%pip install langchain
%pip install pygame
%pip install tiktoken
%pip install mlflow



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 7.2 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached cryptography-46.0.7-cp311-abi3-macosx_10_9_universal2.whl.metadata (5.7 kB)
Using cached cryptography-46.0.7-cp311-abi3-macosx_10_9_universal2.whl (7.2 MB)
  Attempting uninstall: c

In [2]:
import os
from dotenv import load_dotenv
from databricks_langchain import ChatDatabricks
from langchain_core.messages import SystemMessage, HumanMessage as LCHumanMessage


/Users/muskanzehra/Desktop/24280060/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

os.environ.pop("DATABRICKS_HOST", None)
os.environ.pop("DATABRICKS_TOKEN", None)
os.environ.pop("DATABRICKS_AUTH_TYPE", None)
os.environ.pop("DATABRICKS_METADATA_SERVICE_URL", None)
os.environ.pop("DATABRICKS_SERVERLESS_COMPUTE_ID", None)

'auto'

In [7]:

load_dotenv()

print("DATABRICKS_HOST:", "SET" if os.getenv("DATABRICKS_HOST") else "NOT SET")
print("DATABRICKS_TOKEN:", "SET" if os.getenv("DATABRICKS_TOKEN") else "NOT SET")

os.environ["DATABRICKS_AUTH_TYPE"] = "pat"

print("HOST:", os.getenv("DATABRICKS_HOST"))
print("TOKEN:", "SET" if os.getenv("DATABRICKS_TOKEN") else "NOT SET")
print("AUTH:", os.getenv("DATABRICKS_AUTH_TYPE"))

llm = ChatDatabricks(
    endpoint="databricks-qwen3-next-80b-a3b-instruct", # replaced this due to rate limits "databricks-meta-llama-3.1-405b-instruct",
    temperature=0.3,
    max_tokens=4096,
)

def call_llm(system_prompt: str, user_prompt: str, max_tokens: int = 4096) -> str:
    """Call Mosaic AI via ChatDatabricks and return the text reply."""
    messages = [
        SystemMessage(content=system_prompt),
        LCHumanMessage(content=user_prompt),
    ]
    response = llm.invoke(messages)
    return response.content

DATABRICKS_HOST: SET
DATABRICKS_TOKEN: SET
HOST: https://dbc-2a9eaf8c-5dc9.cloud.databricks.com
TOKEN: SET
AUTH: pat


In [8]:
import os
import re
import json
import time
import tempfile
import mlflow
import subprocess
from typing import Callable
from dataclasses import dataclass
from typing import TypedDict, Annotated, List, Optional
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
import tiktoken

# Token counter used by ConversationSummarizationMiddleware (Task 3.1)
TOKENIZER = tiktoken.get_encoding('cl100k_base')

def count_tokens(text: str) -> int:
    return len(TOKENIZER.encode(text))


##Task 1: Initializing System Memory and State Management [10 marks]

In [9]:
def add_strings(str1: List[str], str2: List[str]) -> List[str]:
    """list append reducer just like add_messages but for strings"""
    if str1 is None:
        str1 = []
    if str2 is None:
        return str1
    return str1 + str2


In [10]:
class GameState(TypedDict):
    director_messages: Annotated[List[BaseMessage], add_messages]
    architect_messages: Annotated[List[BaseMessage], add_messages]
    engineer_code: Annotated[List[str], add_strings]
    qa_feedback: Annotated[List[str], add_strings]
    current_actor: str
    iteration: int
    iteration_score: Annotated[List[float], add_strings]
    file_saved: bool
    #--------------------------my PA5 additions------------------------------
    pii_redacted: bool # was PII found and redacted? (2.1)
    hitl_feedback: str # Director feedback injected on rejection(2.2)
    architect_summary: str # compressed architect history(3.1)
    qa_summary: str #compressed QA history(3.1)
    mlflow_run_id: str #active MLflow run id(4)


## PA5 Task 2.1 – PIIMiddleware (Guardrail)
Redacts emails, passwords and API keys from the director prompt.
Logs every redaction event to `guardrail_report.json`.

In [11]:
PII_PATTERNS = {
    'email':    r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+',
    'password': r'(?i)password\s*[:=]\s*\S+',
    'api_key':  r'(?i)(api[_-]?key|token|secret)\s*[:=]\s*[A-Za-z0-9_\-]{8,}',
}

GUARDRAIL_LOG: list = []  # accumulated for JSON report deliverable

def pii_middleware(prompt: str) -> tuple[str, bool]:
    """
    GUARDRAIL- PIIMiddleware
    Redacts PII from the prompt and appends a structured log entry.
    Returns (redacted_prompt, was_anything_redacted).
    """
    redacted = prompt
    findings: list = []

    for pii_type, pattern in PII_PATTERNS.items():
        matches = re.findall(pattern, redacted)
        if matches:
            for m in matches:
                redacted = redacted.replace(m, f'[REDACTED-{pii_type.upper()}]')
            findings.append({'type': pii_type, 'count': len(matches)})

    if findings:
        log_entry = {
            'guardrail': 'PIIMiddleware',
            'action': 'redacted',
            'original_length': len(prompt),
            'redacted_length': len(redacted),
            'findings': findings,
            'redacted_prompt': redacted,
        }
        GUARDRAIL_LOG.append(log_entry)
        print(f'[GUARDRAIL PII] Redacted: {[f["type"] for f in findings]}')

    return redacted, bool(findings)

print('PIIMiddleware has been defined')


PIIMiddleware has been defined


## PA5 Task 1.2 – CodeInterpreterTool & ReAct Helper
The Engineer ReAct agent calls this tool to test code before forwarding.
On failure the agent self-corrects (Task 1.3 – Tool Fallback).

In [12]:
@tool
def code_interpreter_tool(python_code: str) -> str:
    """ Execute Python code in a sandboxed subprocess.
    Returns JSON with keys 'success', 'stdout', 'stderr'.
    On failure the Engineer reads 'stderr' and fixes the code.
    Args: python_code: Complete Python source to execute.
    """
    with tempfile.NamedTemporaryFile(suffix='.py', mode='w', delete=False) as f:
        f.write(python_code)
        fname = f.name
    try:
        result = subprocess.run(
            ['python', fname],
            capture_output=True, text=True, timeout=15
        )
        return json.dumps({
            'success': result.returncode == 0,
            'stdout': result.stdout[:2000],
            'stderr': result.stderr[:2000],
        })
    except subprocess.TimeoutExpired:
        return json.dumps({'success': False, 'stdout': '', 'stderr': 'Execution timed out (15s)'})
    except Exception as e:
        return json.dumps({'success': False, 'stdout': '', 'stderr': str(e)})
    finally:
        os.unlink(fname)

# ReAct agent loop helper  (PA5 Task 1.1)

MAX_REACT_STEPS = 5

def run_react_agent(
    agent_name: str,
    system_prompt: str,
    user_message: str,
    history: list,
    tools: list = [],
) -> str:
    """ ReAct agent loop.Runs Thought-Action-Observation until the model stops calling tools or MAX_REACT_STEPS is reached. Returns the final text response."""
    from langchain_core.messages import SystemMessage as SM
    llm_with_tools = llm.bind_tools(tools) if tools else llm
    tool_map = {t.name: t for t in tools}

    messages = [SM(content=system_prompt)] + history + [HumanMessage(content=user_message)]

    for step in range(MAX_REACT_STEPS):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # If tool calls exist, execute them (Observation step)
        if hasattr(response, 'tool_calls') and response.tool_calls:
            for tc in response.tool_calls:
                fn = tool_map.get(tc['name'])
                obs = fn.invoke(tc['args']) if fn else f'[ERROR] Unknown tool: {tc["name"]}'
                messages.append(HumanMessage(content=f'[Tool result: {tc["name"]}]\n{obs}'))
            print(f'[ReAct {agent_name}] Step {step+1}: tool call(s) executed')
        else:
            print(f'[ReAct {agent_name}] Final response at step {step+1}')
            return response.content

    return messages[-1].content if hasattr(messages[-1], 'content') else ''

print('CodeInterpreterTool & ReAct helper has been defined')


CodeInterpreterTool & ReAct helper has been defined


## PA5 Task 3.1 – Conversation Summarization Middleware
Compresses message history when token count exceeds threshold,
retaining only the essential state of the game.

In [13]:

TOKEN_THRESHOLD = 1500  # compress once history exceeds 1500 tokens

def summarize_messages(messages: List[BaseMessage], label: str) -> List[BaseMessage]:
    """ MIDDLEWARE- ConversationSummarizationMiddleware (Task 3.1) If total tokens exceed TOKEN_THRESHOLD, compress the list into a single summary AIMessage so context stays manageable. """
    if not messages:
        return messages

    full_text = ' '.join(
        m.content for m in messages if hasattr(m, 'content')
    )
    token_count = count_tokens(full_text)

    if token_count <= TOKEN_THRESHOLD:
        return messages  # no summarization needed

    print(f'[MIDDLEWARE Summarize] {label}: {token_count} tokens > {TOKEN_THRESHOLD} → compressing…')

    summary_prompt = (
        f'Summarize the following {label} history for a Dino Runner game pipeline.'
        'Keep only the essential state: agreed requirements, existing code features, '
        'and outstanding issues. Max 250 words.\n\n' + full_text
    )
    summary = call_llm('You are a concise technical summarizer.', summary_prompt, max_tokens=400)
    return [AIMessage(content=f'[SUMMARY – {label}]\n{summary}')]

print('ConversationSummarizationMiddleware has been defined')


ConversationSummarizationMiddleware has been defined


## Task 2: Implement Agent Nodes [40 marks]

### Subtask 2.1: Director and Architect Nodes

In [14]:
def director_node(state: GameState):
    iteration = state.get('iteration', 0)

    if iteration == 0:
        director_msg = input('Awaiting Director Prompt: ').strip()
        if not director_msg:
            director_msg = (
                'Build a Chrome-style Dino Runner game in Python using pygame. '
                'Include a jumping dinosaur, ground cacti, flying pterodactyls, '
                'accurate physics, duck mechanics, and a high-score tracker.'
            )
        print(f'Director Goal recorded: {director_msg[:100]}...')
    else:
        print('Director Using existing goal from state.')
        director_msg = None

    #-------------------------GUARDRAIL: PIIMiddleware--------------------------

    update = {'current_actor': 'director', 'pii_redacted': False}

    if director_msg is not None:
        clean_msg, was_redacted = pii_middleware(director_msg)
        update['director_messages'] = [HumanMessage(content=clean_msg)]
        update['pii_redacted'] = was_redacted

    #------------- MLflow run for this pipeline execution(Task 4)-----------
    mlflow.set_experiment('PA5_DinoRunner')
    run = mlflow.start_run(run_name=f'iteration_{iteration}', nested=True)
    mlflow.log_param('iteration', iteration)
    update['mlflow_run_id'] = run.info.run_id
    mlflow.end_run()

    return update


In [17]:
def architect_node(state: GameState):
    """Architect ReAct Agent(task 1.1). Pure reasoning agent (no tools). Incorporates HITL feedback (Task 2.2) if the Director rejected the previous iteration.
    ConversationSummarizationMiddleware applied before invocation (task 3.1)"""
    iteration = state.get('iteration', 0)
    print(f'\n========== ITERATION {iteration} ==========')

    director_messages = state.get('director_messages', [])
    director_goal = (
        director_messages[0].content
        if director_messages
        else 'Build a Dino Runner game using pygame.'
    )

    # Task 3.1: Summarize architect history if it has gotten bigger
    arch_msgs = state.get('architect_messages', [])
    arch_msgs = summarize_messages(arch_msgs, 'Architect')

    # Task 2.2: Incorporate HITL Director feedback if present
    hitl_feedback = state.get('hitl_feedback', '')
    hitl_context = ''
    if hitl_feedback:
        hitl_context = f'\n\n[Director Feedback to address]:\n{hitl_feedback}'
        print(f'[Architect] Incorporating HITL feedback: {hitl_feedback[:80]}…')

    qa_feedback_list = state.get('qa_feedback', [])
    qa_context = ''
    if qa_feedback_list:
        qa_context = f'\n\n----Previous QA Feedback:\n{qa_feedback_list[-1]}'

    system_prompt = (
        'You are a senior software architect specialising in Python game development. '
        'Reason step-by-step (Thought) then produce a detailed technical design '
        'document (Action). Cover: class structure, game loop, physics, scoring, '
        'day/night transition, clustered obstacles, speed scaling, persistent high score.'
    )
    user_message = (
        f'Director Goal:\n{director_goal}'
        f'{qa_context}{hitl_context}\n\n'
        'Produce the full technical design document now.'
    )

    t_start = time.time()
    print('[Architect] Running ReAct reasoning…')

    # ReAct loop-pure reasoning, no tools for Architect (task 1.1)
    design_doc = run_react_agent(
        agent_name='Architect',
        system_prompt=system_prompt,
        user_message=user_message,
        history=arch_msgs,
        tools=[],
    )
    latency = time.time() - t_start
    print(f'[Architect] Design ready ({len(design_doc)} chars) in {latency:.1f}s.')

    # task 4: Log architect latency to MLflow
    run_id = state.get('mlflow_run_id', '')
    if run_id:
        with mlflow.start_run(run_id=run_id, nested=True):
            mlflow.log_metric('architect_latency_s', latency, step=iteration)

    return {
        'architect_messages': [AIMessage(content=design_doc)],
        'current_actor': 'architect',
        'iteration': iteration + 1,
        'file_saved': False,
        'hitl_feedback': '',  # clear after consuming
    }


### Subtask 2.2: Engineer Node

In [18]:
import re

def _extract_code(raw: str) -> str:
    match = re.search(r'```(?:python)?\n(.*?)```', raw, re.DOTALL)
    return match.group(1).strip() if match else raw.strip()


def engineer_node(state: GameState) -> dict:
    """ Engineer ReAct Agent (PA5 tasks 1.1, 1.2, 1.3).
    Uses CodeInterpreterTool to test its code before forwarding.
    On tool failure, the agent reads stderr and self-corrects (task 1.3). """
    architect_messages = state.get('architect_messages', [])
    architect_message = (
        architect_messages[-1].content
        if architect_messages
        else 'No architect design available.'
    )

    prev_code_list = state.get('engineer_code', [])
    prev_code = prev_code_list[-1] if prev_code_list else None

    qa_messages_list = state.get('qa_feedback', [])
    qa_messages = qa_messages_list[-1] if qa_messages_list else None

    refinement_context = ''
    if prev_code and qa_messages:
        refinement_context = (
            f'\n\n----PREVIOUS CODE (iteration {len(prev_code_list)})----\n'
            f'{prev_code}\n'
            f'\n-----QA FEEDBACK for refinement------\n{qa_messages}\n'
            'Please fix all issues reported above.'
        )

    system_prompt = (
        'You are a senior Python game developer using a ReAct approach.\n'
        'WORKFLOW (Task 1.1 ReAct):\n'
        '1. Thought: plan the implementation.\n'
        '2. Action: call code_interpreter_tool with a short SYNTAX-CHECK snippet '
           '(e.g. import statements + class stubs + sys.exit(0)).\n'
        '3. Observation: if success=false, read stderr, fix, and call the tool again '
           '(Tool Fallback – Task 1.3). Repeat until success=true.\n'
        '4. Final Action: return the FULL complete game source code as plain text.\n\n'
        'Requirements for the game:\n'
        '  • Headless when DISPLAY not set (pygame.NOFRAME).\n'
        '  • dinosaur jump + duck, ground cacti, flying pterodactyls, score, high-score.\n'
        '  • Day/night transition, clustered obstacles, speed scaling.\n'
        '  • Output ONLY the raw Python code — no markdown, no explanation.'
    )
    user_message = (
        f'Architecture Design:\n{architect_message}'
        f'{refinement_context}'
    )

    t_start = time.time()
    print('[Engineer] Running ReAct + CodeInterpreterTool…')

    # ReAct loop with CodeInterpreterTool (Tasks 1.1, 1.2, 1.3)
    raw_output = run_react_agent(
        agent_name='Engineer',
        system_prompt=system_prompt,
        user_message=user_message,
        history=[],
        tools=[code_interpreter_tool],  # Task 1.2
    )
    latency = time.time() - t_start
    clean_code = _extract_code(raw_output)
    print(f'[Engineer] Code ready ({len(clean_code)} chars) in {latency:.1f}s.')

    # task 4: Log engineer latency to MLflow
    run_id = state.get('mlflow_run_id', '')
    if run_id:
        with mlflow.start_run(run_id=run_id, nested=True):
            mlflow.log_metric('engineer_latency_s', latency, step=state.get('iteration', 0))

    return {
        'engineer_code': [clean_code],
        'current_actor': 'engineer',
    }


### Subtask 2.3: File I/O and Execution Nodes

In [19]:
def file_writer(state: GameState):

    code_list = state.get("engineer_code", [])
    code = code_list[-1] if code_list else ""    

    if not code:
        print("No code to save- File writer")
        return {"file_saved": False}

    filepath = "dino_runner.py"
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(code)

    print(code[:500])
    print("\n[File saved: dino_runner.py]")

    return {"file_saved": True, "current_actor": "file_writer"}


def run_code(state: GameState):
    print("\n===== CODE EXECUTION =====")
    
    if not state.get("file_saved", False):
        print("file not saved so skipping execution")
        return {"qa_feedback": ["EXECUTION SKIPPEd- File was not saved."], "current_actor": "run_code"}
    
    choice = input("Run the generated game? (y/n): ").strip().lower()
    
    if choice != "y":
        print("Execution skipped by user.")
        run_output = "EXECUTION HAS BEENSKIPPED BY THE USER"
    else:
        print("RunCode-Launching dino_runner.py")
        try:
            result = subprocess.run(
                ["python", "dino_runner.py"],
                capture_output=True,
                text=True,
                timeout=30, 
            )
            run_output = (
                f"STDOUT:\n{result.stdout}\n"
                f"STDERR:\n{result.stderr}\n"
                f"Return code:{result.returncode}"
            )
        except subprocess.TimeoutExpired:
            run_output = "PROCESS TIMED OUT"
        except Exception as exc:
            run_output = f"EXECUTION FAILED - MSG: {exc}"

        print(run_output[:300])

    return {
        "qa_feedback": [f"RUN_OUTPUT-\n{run_output}"],
        "current_actor": "run_code",
    }

### Subtask 2.4: QA and Scorer Nodes

In [20]:
def qa_node(state: GameState):
    """QA Engineer ReAct Agent (task 1.1).
    Pure reasoning -reviews code against design requirements.
    task 3.1: qa history summarized when token threshold exceeded.
    task 4: latency logged to MLflow. """
    
    # task 3.1: Summarize QA history before building context
    qa_feedback_list = list(state.get('qa_feedback', []))
    
    # Convert qa strings to BaseMessage list for summarization
    qa_msgs_for_summary = [AIMessage(content=m) for m in qa_feedback_list]
    qa_msgs_for_summary = summarize_messages(qa_msgs_for_summary, 'QA')
    run_output = qa_msgs_for_summary[-1].content if qa_msgs_for_summary else 'NO EXECUTION OUTPUT'

    architect_msgs = state.get('architect_messages', [])
    design_requirements = (
        architect_msgs[-1].content if architect_msgs else 'NO DESIGN REQUIREMENTS BY ARCHITECT'
    )

    code_list = state.get('engineer_code', [])
    engineer_code = code_list[-1] if code_list else 'NO CODE GENERATED BY ENGINEER'

    system_prompt = (
        'You are an expert QA engineer using a ReAct approach. '
        'Reason step by step (Thought) then produce your report (Action). '
        'Analyse the Python game code against the design requirements and '
        'the execution output. Report: 1:syntax errors, 2:runtime errors, '
        '3:missing features, 4:logic bugs, 5:recommended fixes. '
        'Also assign a score 1-10 at the end as: SCORE: <n>'
    )
    user_prompt = (
        f'----DESIGN REQUIREMENTS----\n{design_requirements[:2000]}\n\n'
        f'----ENGINEER CODE----\n{engineer_code[:3000]}\n\n'
        f'----EXECUTION OUTPUT----\n{run_output[:1000]}'
    )

    t_start = time.time()
    print('[QA] Running ReAct analysis…')

    # ReAct loop - pure reasoning, no tools for QA 
    qa_analysis = run_react_agent(
        agent_name='QA',
        system_prompt=system_prompt,
        user_message=user_prompt,
        history=[],
        tools=[],
    )
    latency = time.time() - t_start

    # task 4: Log QA latency to MLflow
    run_id = state.get('mlflow_run_id', '')
    if run_id:
        with mlflow.start_run(run_id=run_id, nested=True):
            mlflow.log_metric('qa_latency_s', latency, step=state.get('iteration', 0))

    return {
        'qa_feedback': [qa_analysis],
        'current_actor': 'qa',
    }


def score_node(state: GameState):
    """Scorer Node. task 4: logs QA score AND groundedness (LLM-as-a-judge) to MLflow."""
    qa_feedback_list = state.get('qa_feedback', [])
    qa_feedback = qa_feedback_list[-1] if qa_feedback_list else 'NO QA FEEDBACK.'

    system_prompt = (
        'You are a code quality assessor. Given a QA report about a Python game, '
        'assign a single integer score from 1 (totally broken) to 10 (perfect). '
        'Reply with ONLY the integer — no other text.'
    )
    user_prompt = f'QA Report:\n{qa_feedback}'

    print('COMPUTING QUALITY SCORE USING LLM')
    raw_score = call_llm(system_prompt, user_prompt, max_tokens=10).strip()
    try:
        score = float(re.search(r'\d+(\.\d+)?', raw_score).group())
        score = max(1.0, min(10.0, score))
    except Exception:
        score = 5.0

    print(f'Score for iteration {state.get("iteration", "?")}: {score}/10')

    # task 4: Groundedness - LLM-as-a-judge
    # Measures how accurately the Engineer code reflects the Architect plan.
    arch_msgs = state.get('architect_messages', [])
    arch_plan = arch_msgs[-1].content if arch_msgs else ''
    code_list = state.get('engineer_code', [])
    eng_code  = code_list[-1] if code_list else ''

    ground_prompt = (
        'Rate 0-100 how accurately the Python code below implements the '
        'architecture plan. Reply with ONLY a number.\n\n'
        f'PLAN:\n{arch_plan[:800]}\n\nCODE:\n{eng_code[:1500]}'
    )
    ground_raw = call_llm('You are a strict technical judge.', ground_prompt, max_tokens=10).strip()
    try:
        groundedness = float(re.search(r'\d+(\.\d+)?', ground_raw).group())
        groundedness = max(0.0, min(100.0, groundedness))
    except Exception:
        groundedness = 50.0

    print(f'Groundedness score: {groundedness}/100')

    # task 4: Log both metrics to MLflow
    run_id = state.get('mlflow_run_id', '')
    iteration = state.get('iteration', 0)
    if run_id:
        with mlflow.start_run(run_id=run_id, nested=True):
            mlflow.log_metric('qa_score', score,step=iteration)
            mlflow.log_metric('groundedness',groundedness, step=iteration)
            mlflow.log_metric('iteration',iteration,step=iteration)

    return {
        'iteration_score': [score],
        'current_actor': 'scorer',
    }


In [21]:
def should_continue(state: GameState) -> str:
    """ HITL Approval Node (pa5 tsk 2.2).
    Director can: approve (end), OR reject and inject custom feedback
    which routes the graph back to Architect for the next iteration."""
    scores = state.get('iteration_score', [])
    latest_score = scores[-1] if scores else 0
    iteration    = state.get('iteration', 0)

    print('\n' + '='*50)
    print(f'----Current score----: {latest_score}/10')
    print(f'----Iteration--------: {iteration}')
    print('='*50)

    # task 2.2: HITL: Director decides to approve or reject
    # every iteration has a HITL pause here.
    choice = input('Do you want to refine the code? (y/n): ').strip().lower()

    if choice == 'y':
        # task 2.2: Director can inject custom feedback back to Architect
        feedback = input(
            'Enter Director feedback for Architect (or press Enter to skip): '
        ).strip()
        if feedback:
            print(f'[HITL] Director feedback injected: {feedback[:80]}…')
            # Inject into state via a side-effect update (LangGraph pattern)
            state['hitl_feedback'] = feedback
        else:
            state['hitl_feedback'] = ''
        print('Routing back to Architect for revision.')
        return 'architect'
    else:
        # task 5: Save guardrail report on successful completion
        if GUARDRAIL_LOG:
            with open('guardrail_report.json', 'w') as f:
                json.dump(GUARDRAIL_LOG, f, indent=2)
            print('Guardrail report saved → guardrail_report.json')
        print('Workflow complete - routing to end.')
        return 'end'


## Task 3: Create Graph Structure [20 marks]

In [22]:
builder = StateGraph(GameState)
builder.add_node('director', director_node)
builder.add_node('architect', architect_node)
builder.add_node('engineer', engineer_node)
builder.add_node('file_writer', file_writer)
builder.add_node('run_code', run_code)
builder.add_node('qa', qa_node)
builder.add_node('scorer', score_node)

builder.add_edge(START, 'director')
builder.add_edge('director', 'architect')
builder.add_edge('architect', 'engineer')
builder.add_edge('engineer', 'file_writer')
builder.add_edge('file_writer', 'run_code')
builder.add_edge('run_code', 'qa')
builder.add_edge('qa', 'scorer')

# task 2.2: HITL routes back to architect (not engineer) so Director feedback is re-planned before re-coding
builder.add_conditional_edges(
    'scorer',
    should_continue,
    {
        'architect': 'architect',   # go back to architect with feedback
        'end': END,
    },
)

memory = MemorySaver()  #persistent checkpointing
app = builder.compile(checkpointer=memory)

print('GRAPH COMPILED')
print('NODES: ', list(builder.nodes.keys()))


GRAPH COMPILED
NODES:  ['director', 'architect', 'engineer', 'file_writer', 'run_code', 'qa', 'scorer']


## Task 4: System Invocation [10 marks]

In [25]:
config = {'configurable': {'thread_id': 'dino_runner_session_1'}}

initial_state: GameState = {
    'director_messages': [],
    'architect_messages': [],
    'engineer_code': [],
    'qa_feedback': [],
    'current_actor': '',
    'iteration': 0,
    'iteration_score': [],
    'file_saved': False,
    #-----------------------PA5 additions -----------------
    'pii_redacted':False, # task 2.1
    'hitl_feedback': '', # task 2.2
    'architect_summary': '', # task 3.1
    'qa_summary': '', # task 3.1
    'mlflow_run_id':  '', # task 4
}

print('-----------------STARTING WORKFLOW-------------------')

# task 4: wrap everything in a parent MLflow run
mlflow.set_experiment('PA5_DinoRunner')
with mlflow.start_run(run_name='PA5_Parent_Run'):
    mlflow.log_param('student_id', '24280060')
    mlflow.log_param('assignment', 'PA5')

    for event in app.stream(initial_state, config=config, stream_mode='updates'):
        for node, value in event.items():
            print(f'\n===== {node.upper()} =====')
            if isinstance(value, dict):
                for k, v in value.items():
                    if isinstance(v, list) and v:
                        preview = str(v[-1])[:120].replace('\n', ' ')
                        print(f'  {k} (latest): {preview} …')
                    elif isinstance(v, str) and len(v) > 100:
                        print(f'  {k}: {v[:120]} …')
                    else:
                        print(f'  {k}: {v}')

print('\n--------------WORKFLOW COMPLETE-----------------')


-----------------STARTING WORKFLOW-------------------


{"ts": "2026-05-23 16:42:11.423", "level": "ERROR", "logger": "pyspark.sql.connect.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_InactiveRpcError", "msg": "<_InactiveRpcError of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_status:13, grpc_message:\"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"}\"\n>", "stacktrace": [{"class": null, "method": "config", "file": "/Users/muskanzehra/Desktop/24280060/.venv/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py", "line": "2102"}, {"class": null, "method": "__call__", "file": "/Users/muskanzehra/Desktop/24280060/.venv/lib/python3.12/site-packages/grpc/_interceptor.py", "line": "276"}, {"class": null, "method": "_with_call", "file": "/Users/muskanzehra

Director Goal recorded: Build a Chrome-style Dino Runner game using Python and pygame with the following specifications:  VI...


{"ts": "2026-05-23 16:42:24.094", "level": "ERROR", "logger": "pyspark.sql.connect.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_InactiveRpcError", "msg": "<_InactiveRpcError of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_status:13, grpc_message:\"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"}\"\n>", "stacktrace": [{"class": null, "method": "config", "file": "/Users/muskanzehra/Desktop/24280060/.venv/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py", "line": "2102"}, {"class": null, "method": "__call__", "file": "/Users/muskanzehra/Desktop/24280060/.venv/lib/python3.12/site-packages/grpc/_interceptor.py", "line": "276"}, {"class": null, "method": "_with_call", "file": "/Users/muskanzehra


===== DIRECTOR =====
  current_actor: director
  pii_redacted: False
  director_messages (latest): content='Build a Chrome-style Dino Runner game using Python and pygame with the following specifications:  VISUALS & COL …
  mlflow_run_id: b63db8c4614c44b78aa8425ccd287ffb

========== ITERATION 0 ==========
[MIDDLEWARE Summarize] Architect: 22112 tokens > 1500 → compressing…
[Architect] Running ReAct reasoning…
[ReAct Architect] Final response at step 1
[Architect] Design ready (14812 chars) in 35.6s.


...


===== QA =====
  qa_feedback (latest): Thought:   The provided code is incomplete and contains critical syntax and structural errors that prevent it from being …
  current_actor: qa
COMPUTING QUALITY SCORE USING LLM
Score for iteration 2: 3.0/10
Groundedness score: 0.0/100


{"ts": "2026-05-23 16:52:03.335", "level": "ERROR", "logger": "pyspark.sql.connect.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_InactiveRpcError", "msg": "<_InactiveRpcError of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_message:\"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\", grpc_status:13}\"\n>", "stacktrace": [{"class": null, "method": "config", "file": "/Users/muskanzehra/Desktop/24280060/.venv/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py", "line": "2102"}, {"class": null, "method": "__call__", "file": "/Users/muskanzehra/Desktop/24280060/.venv/lib/python3.12/site-packages/grpc/_interceptor.py", "line": "276"}, {"class": null, "method": "_with_call", "file": "/Users/muskanzehra


----Current score----: 3.0/10
----Iteration--------: 2
Guardrail report saved → guardrail_report.json
Workflow complete - routing to end.

===== SCORER =====
  iteration_score (latest): 3.0 …
  current_actor: scorer


{"ts": "2026-05-23 16:52:14.051", "level": "ERROR", "logger": "pyspark.sql.connect.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_InactiveRpcError", "msg": "<_InactiveRpcError of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_message:\"[CONFIG_NOT_AVAILABLE] Configuration spark.mlflow.modelRegistryUri is not available. SQLSTATE: 42K0I\", grpc_status:13}\"\n>", "stacktrace": [{"class": null, "method": "config", "file": "/Users/muskanzehra/Desktop/24280060/.venv/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py", "line": "2102"}, {"class": null, "method": "__call__", "file": "/Users/muskanzehra/Desktop/24280060/.venv/lib/python3.12/site-packages/grpc/_interceptor.py", "line": "276"}, {"class": null, "method": "_with_call", "file": "/Users/muskanzehra


--------------WORKFLOW COMPLETE-----------------


## My prompt

Build a Chrome-style Dino Runner game using Python and pygame with the following specifications:

VISUALS & COLORS:
- Sky background: light blue (#87CEEB) that darkens to purple (#2C1654) at night mode after score 500
- Ground: solid brown bar (#8B4513) at bottom with green grass strip (#228B22) on top
- Dinosaur: green (#2ECC71) rectangle body with darker green (#27AE60) legs, white eye (#FFFFFF) with black pupil, gray duck shape when ducking
- Cacti: dark green (#1A5C1A) with multiple arms, thick trunks, arranged in groups of 1 to 3
- Pterodactyls: orange (#E67E22) bird shape with two wing positions (flapping animation), flies at 3 different heights
- Score text: bold black font top right corner with golden (#FFD700) high score beside it
- Clouds: white (#FFFFFF) slow-moving puffs in the background
- Game Over screen: red GAME OVER text centered, with PRESS SPACE TO RESTART below it

GAMEPLAY:
- Dinosaur jumps with SPACE or UP arrow, double jump allowed
- Dinosaur ducks with DOWN arrow, hitbox shrinks when ducking
- Gravity pulls dinosaur down realistically with acceleration
- Game speed increases every 100 points
- Obstacles spawn randomly with minimum safe gap between them
- High score persists during the session and shown at top
- Score increments every frame and displays as integer

Make the code complete, single file, and immediately runnable with no missing assets.

## Prompt with pii 

Build a Chrome-style Dino Runner game using Python and pygame with the following specifications:

VISUALS & COLORS:
- Sky background: light blue (#87CEEB) that darkens to purple (#2C1654) at night mode after score 500
- Ground: solid brown bar (#8B4513) at bottom with green grass strip (#228B22) on top
- Dinosaur: green (#2ECC71) rectangle body with darker green (#27AE60) legs, white eye (#FFFFFF) with black pupil, gray duck shape when ducking
- Cacti: dark green (#1A5C1A) with multiple arms, thick trunks, arranged in groups of 1 to 3
- Pterodactyls: orange (#E67E22) bird shape with two wing positions (flapping animation), flies at 3 different heights
- Score text: bold black font top right corner with golden (#FFD700) high score beside it
- Clouds: white (#FFFFFF) slow-moving puffs in the background
- Game Over screen: red GAME OVER text centered, with PRESS SPACE TO RESTART below it

GAMEPLAY:
- Dinosaur jumps with SPACE or UP arrow, double jump allowed
- Dinosaur ducks with DOWN arrow, hitbox shrinks when ducking
- Gravity pulls dinosaur down realistically with acceleration
- Game speed increases every 100 points
- Obstacles spawn randomly with minimum safe gap between them
- High score persists to disk across multiple launches using JSON file
- Score increments every frame and displays as integer
- Day/Night sky transition after score 500
- Clustered obstacles: 2-3 cacti spawn together

Make the code complete, single file, and immediately runnable with no missing assets.

Developer contact: dev@lums.edu.pk
api_key=sk-test123456789ABCDEF password: secret123